# ZaramTech AB Website — Build Log Notebook

This notebook is a **step-by-step build log**. It records, layer by layer:

1. The **command** that was run (and confirmed working).
2. The **reason** it was used.
3. What to **expect / verify** afterward.

> How to use it: run the cells in order from top to bottom. Each command cell uses the shell (`!`) so it runs against your system exactly as it did during setup.
>
> **Environment confirmed working:**
> - Node.js `v24.16.0`
> - npm `11.17.0`
> - OS: Linux
>
> Project location: `/home/a476078/WebApp/zaramtech-website`

## Layer 0 — Verify prerequisites

Before building anything, confirm the toolchain exists. If these print version numbers, you're good. If any command is "not found", install it first.

In [ ]:
# Command: check the installed tool versions
# Reason: Next.js requires Node.js (LTS). Confirming Node, npm, and Git are
#         present up front avoids confusing failures later in the build.
!node --version
!npm --version
!git --version

## Layer 1 — Scaffold the Next.js foundation

This creates the whole project skeleton: TypeScript, Tailwind CSS, ESLint, the App Router, a `src/` directory, and the `@/*` import alias.

> **Note:** This was already run to create the project. You do **not** need to run it again — it is documented here so you understand exactly how the foundation was produced. Re-running it in an existing folder will fail or overwrite.

In [ ]:
# Command (already executed — shown for the record, do NOT re-run):
#
#   npx --yes create-next-app@latest zaramtech-website \
#       --typescript --tailwind --eslint --app \
#       --src-dir --import-alias "@/*" --no-turbopack --use-npm
#
# Reason for each flag:
#   --typescript      -> type-safe, maintainable code (fewer runtime bugs)
#   --tailwind        -> utility-first CSS so we can apply the design tokens fast
#   --eslint          -> catches code-quality/style issues automatically
#   --app             -> App Router: modern Next.js routing + layouts + metadata
#   --src-dir         -> keeps source under src/ so root stays clean
#   --import-alias @/*-> lets us import with "@/components/..." instead of ../../
#   --no-turbopack    -> use the stable webpack build for predictability
#   --use-npm         -> pin the package manager to npm (matches our lockfile)
print("Layer 1 already complete. See package.json below for the result.")

In [ ]:
# Command: inspect what the scaffold produced
# Reason: Verify the dependencies (next, react, tailwind, typescript) and the
#         npm scripts (dev/build/start/lint) were set up as expected.
!cd /home/a476078/WebApp/zaramtech-website && cat package.json

In [ ]:
# Command: list the generated source tree
# Reason: Confirm the App Router files exist. After scaffolding you should see:
#         src/app/layout.tsx  (root layout — wraps every page)
#         src/app/page.tsx    (the home page)
#         src/app/globals.css (global styles + Tailwind)
!cd /home/a476078/WebApp/zaramtech-website && find src -type f

## Layer 2 — Run the dev server (smoke test)

Before customizing anything, confirm the untouched scaffold actually runs. This proves the foundation is healthy.

> The dev server is a **long-running process** — it does not exit on its own. In a notebook, run it in a real terminal instead (`npm run dev`) so it doesn't block the notebook. The cell below shows the command and reason.

In [ ]:
# Command (run this in a TERMINAL, not the notebook, because it stays running):
#
#   cd /home/a476078/WebApp/zaramtech-website && npm run dev
#
# Reason: Starts the local development server at http://localhost:3000 with
#         hot-reload. Seeing the default Next.js page confirms Node, Next,
#         React, and Tailwind are all wired correctly before we build features.
print("Open http://localhost:3000 after running `npm run dev` in a terminal.")

In [ ]:
# Command: produce a production build (a non-blocking way to validate the app)
# Reason: `next build` compiles and type-checks the whole project and exits.
#         Unlike `npm run dev`, it finishes on its own, so it's safe to run in
#         a notebook. A successful build = the foundation is solid.
!cd /home/a476078/WebApp/zaramtech-website && npm run build

## Layer 3 — Verify code quality (lint)

ESLint enforces consistent, bug-resistant code. Run it any time before committing.

In [ ]:
# Command: run the linter
# Reason: Catches unused variables, bad imports, and style issues early so the
#         codebase stays clean as we add layers (components, sections, form).
!cd /home/a476078/WebApp/zaramtech-website && npm run lint

## Layer 4 — Design tokens (brand colors + fonts)

**What this layer does (simple version):** it teaches the project our brand's look — the navy and cyan colors and the two fonts — so every page can reuse them by name (like `bg-navy` or `text-cyan`) instead of repeating hex codes everywhere.

**Why we do it now:** setting the "paint and fonts" once, before building sections, means the Hero, Services, and Contact all stay visually consistent and are easy to restyle later from one place.

### Files changed

**1. `src/app/globals.css` — added the color + font tokens**

This project uses **Tailwind v4**, which defines colors in CSS with an `@theme` block (there is no `tailwind.config.js`). We added:
- Brand colors: `navy` `#1F3A5F`, `cyan` `#00C2CB` (+ dark/light variants)
- Surfaces: white + light gray
- Text: near-black + muted slate
- Fonts wired to the layout, and a `prefers-reduced-motion` rule so animations respect accessibility settings.

*Why:* these are the exact tokens from `docs/03-visual-design-system.md`, now usable as Tailwind classes.

**2. `src/app/layout.tsx` — loaded the fonts and set the page title**

- Loads **Inter** (body) and **Space Grotesk** (headings) via `next/font` — this self-hosts the fonts for speed and zero layout shift.
- Set the real page **title** and **description** (SEO + browser tab).

*Why:* headings in a geometric sans give the "engineering + AI" feel; `next/font` keeps the site fast, which matters for a performance brand.


In [ ]:
# Command: rebuild after adding the design tokens
# Reason: A clean build proves the new colors/fonts in globals.css and layout.tsx
#         are valid and didn't break anything. Do this after every layer.
!cd /home/a476078/WebApp/zaramtech-website && npm run build 2>&1 | tail -12


## Layer 5 — Header & Footer (site frame)

**What this layer does (simple version):** it adds the top navigation bar and the bottom footer that appear on every page. The footer also shows the legal details a Swedish AB must display (company name, org.nr, location).

**Why we do it now:** the header and footer are the "frame" around all content. Building them first gives every later section (Hero, Services, Contact) a consistent shell to live inside.

### Clean-code choices (separation of concerns)

We split **data** from **presentation** so each file has one job:

| File | One job |
|------|---------|
| `src/config/site.ts` | **Data only** — company name, org.nr, email, nav links, CTA. Edit these facts in ONE place. |
| `src/components/layout/Header.tsx` | **Presentation only** — draws the nav bar; reads links from the config. |
| `src/components/layout/Footer.tsx` | **Presentation only** — draws the footer + legal block; reads from the config. |
| `src/app/layout.tsx` | **Composition** — stacks Header → page → Footer. |

*Why this matters:* to change the menu or the org number later, you touch `site.ts` only — the components never need editing. This keeps things readable and avoids the same text being copy-pasted in many files.

### Details worth knowing
- The header is `sticky` so it stays visible while scrolling.
- Nav uses a real `<nav>` + `<ul>` with `aria-label` — good for accessibility and screen readers.
- External LinkedIn link uses `rel="noopener noreferrer"` for security.
- The footer year is generated with `new Date().getFullYear()` so it never goes stale.
- `org.nr` and `email` are placeholders (`556XXX-XXXX`, `hello@zaramtech.se`) — replace them in `site.ts` with the real values.


In [ ]:
# Command: rebuild after adding the Header, Footer, and site config
# Reason: Confirms the new components and the "@/config/site" import path all
#         resolve and render. A clean build = Layer 5 is wired correctly.
!cd /home/a476078/WebApp/zaramtech-website && npm run build 2>&1 | tail -12


## What's next (upcoming layers)

Progress so far, and what's coming. Each layer is added to this log with its command + reason as we build it:

| Layer | Goal | Status |
|-------|------|--------|
| 4 | **Design tokens** — navy `#1F3A5F` + cyan colors and fonts | ✅ done |
| 5 | **Layout** — `Header` (nav) and `Footer` (org.nr, legal) components | ✅ done |
| 6 | **Hero** — tagline + CTA + subtle technical motif | ⬜ next |
| 7 | **Services** — reusable `ServiceCard` × 5 | ⬜ |
| 8 | **Case studies** — Problem → Approach → Result | ⬜ |
| 9 | **Contact** — GDPR-compliant form + serverless email handler | ⬜ |
| 10 | **SEO** — metadata + Organization JSON-LD | ⬜ |
| 11 | **Deploy** — push to GitHub, deploy on Vercel, connect domain | ⬜ |

> Each time we add a layer, a new section is appended here so this notebook stays a complete, runnable record of how the site was built.
